In [0]:
--Tabla 1: silver_matchs-------------------------------------------------------------------
SELECT
    REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
    name,
    league,
    match[0].awayTeam.name AS away_team,
    match[0].homeTeam.name AS home_team,
    match[0].location.name AS stadium,
    cast(match[0].startDate as TIMESTAMP) AS start_date,
    cast(match[0].endDate as TIMESTAMP) AS end_date,
    match[0].url AS url
FROM 
    futbol.bronze_matchs
ORDER BY
    date DESC;


--Tabla 2: silver_match_events-------------------------------------------------------------------
SELECT
    REPLACE(substring_index(match[0].url, '-', -1), "/", "") AS match_id,
    CAST(match[0].subEvent[0].startDate AS TIMESTAMP) AS event_time,
    sub_event.attendee.name AS event_team,
    sub_event.name AS description,
    CASE
        WHEN sub_event.name IS NULL THEN ""
        WHEN sub_event.name LIKE 'Gol%' THEN 'GOL'
        WHEN sub_event.name LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
        WHEN sub_event.name LIKE 'Expulsi%' THEN 'TARJETA ROJA'
        WHEN sub_event.name LIKE 'Sale%' THEN 'CAMBIO'
        ELSE 'OTHER'
    END as event_type
FROM 
    futbol.bronze_matchs
LATERAL VIEW EXPLODE(match[0].subEvent) exploded AS sub_event
ORDER BY
    date DESC;


--Tabla 3: silver_match_players-------------------------------------------------------------------
SELECT
    REPLACE(substring_index(match[0].url, '-', -1), '/', '') AS match_id,
    team.name AS team_name,
    team_players.name AS player_name,
    team_players.roleName AS player_position
FROM
    futbol.bronze_matchs
    LATERAL VIEW EXPLODE(match[1].`@graph`) t AS team
    LATERAL VIEW EXPLODE(team.athlete) p AS team_players
WHERE
    team.`@type` = 'SportsTeam'
ORDER BY
    date DESC;


--Tabla 4: silver_match_referees-------------------------------------------------------------------
SELECT
    REPLACE(substring_index(match[0].url, '-', -1), '/', '') AS match_id,
    referee.name AS name,
    referee.jobTitle AS role
FROM
    futbol.bronze_matchs
    LATERAL VIEW EXPLODE(match[0].`attendee`) t AS referee
ORDER BY
    date DESC;